What makes a good test case set
You want geometries that exercise different failure modes and optimization behaviors:
CaseWhy it's usefulBaseline bracketThe default — confirms the pipeline runs end to endThin wallStresses the mesh quality checker — high aspect ratio elements likelySquat/wideLow height-to-footprint ratio — tests surface classification in gmshHeavy loadPushes safety factor below threshold — tests Stage 3's warning logicLight loadVery low stress — tests SIMP when there's little to optimizeFine meshSmall target_element_size — tests memory and solver at scaleAggressive optimizationLow volume_fraction — tests SIMP convergence under tight constraintsConservative optimizationHigh volume_fraction — tests near-solid result in Stage 5

Cell 0 — Parameters (tag: parameters)

In [1]:
# Cell 0 — tagged: parameters
# Controls where generated params files are written.
# Override via Papermill if needed.

import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")

OUTPUT_DIR     = "test_cases"
PREVIEW_SCAD   = True
OVERWRITE      = False

Cell 1 — Imports and helpers

In [2]:
# Cell 1 — Setup
import json
import copy
import shutil
from pathlib import Path
from dataclasses import asdict

from src.geometry.param_schema import PipelineParams

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Load the baseline params — all test cases are derived from this
BASELINE = json.loads(Path("scad/params.json").read_text())
print("Baseline loaded:")
print(json.dumps(BASELINE, indent=2))


def make_case(name: str, overrides: dict) -> dict:
    """
    Deep-merge overrides into a copy of the baseline params.
    overrides uses the same nested structure as params.json.

    Example:
        make_case("my_case", {
            "geometry": {"length": 150.0},
            "load_hints": {"load_magnitude_n": 2000.0},
        })
    """
    case = copy.deepcopy(BASELINE)
    case["part_name"] = name

    for section, values in overrides.items():
        if isinstance(values, dict) and section in case:
            case[section].update(values)
        else:
            case[section] = values

    return case


def write_case(case: dict) -> Path:
    """
    Validate the case against PipelineParams schema,
    then write it to OUTPUT_DIR/<part_name>/params.json.
    Returns the path written.
    """
    name = case["part_name"]
    case_dir = output_dir / name
    params_path = case_dir / "params.json"

    if params_path.exists() and not OVERWRITE:
        print(f"  ⏭  {name} — already exists, skipping (set OVERWRITE=True to replace)")
        return params_path

    # Validate before writing — catches bad values early
    try:
        params = PipelineParams.from_dict(case)
        params.validate()
    except (AssertionError, KeyError, TypeError) as e:
        print(f"  ✗  {name} — validation failed: {e}")
        return None

    case_dir.mkdir(parents=True, exist_ok=True)

    # Copy scad source into the case directory so each case is self-contained
    shutil.copy("scad/base_part.scad", case_dir / "base_part.scad")

    # Update stl_output_dir to point to this case's own outputs folder
    case["export"]["stl_output_dir"] = str(case_dir / "outputs" / "meshes")

    params_path.write_text(json.dumps(case, indent=2))
    print(f"  ✓  {name} → {params_path}")
    return params_path

Baseline loaded:
{
  "_comment": "Default parameters for base_part.scad. Overridden at runtime by Papermill.",
  "part_name": "base_part",
  "geometry": {
    "length": 100.0,
    "width": 60.0,
    "height": 20.0,
    "wall_thickness": 4.0,
    "fillet_radius": 2.0,
    "mounting_hole_diameter": 6.0,
    "mounting_hole_inset": 10.0
  },
  "mesh_hints": {
    "target_element_size": 8.0,
    "opt_domain_element_size_mm": 2.5,
    "refinement_regions": []
  },
  "load_hints": {
    "primary_face": "top",
    "load_magnitude_n": 10000.0
  },
  "boundary_conditions": {
    "fixed_face": "corners",
    "load_face": "top",
    "load_direction": [
      0.0,
      0.0,
      -1.0
    ],
    "hole_inset_fraction": 0.2,
    "shell_thickness_mm": 2.0
  },
  "export": {
    "stl_output_dir": "outputs/meshes",
    "stl_ascii": false
  }
}


Cell 2 — Define test cases

In [3]:
# Cell 2 — Test case definitions
#
# Each case is a name + a dict of overrides against the baseline.
# Only specify what changes — everything else inherits from scad/params.json.
#
# Cases are grouped by what pipeline behavior they exercise.

TEST_CASES = [

    # ── Group A: Geometry variations ──────────────────────────────────────
    # Tests that OpenSCAD and gmsh handle different proportions correctly.

    make_case("baseline", {}),   # Unmodified — confirms basic pipeline runs

    make_case("tall_narrow", {
        "geometry": {
            "length": 60.0,
            "width":  40.0,
            "height": 50.0,    # tall relative to footprint
        },
        "_comment": "High aspect ratio geometry — tests surface classification "
                    "in gmsh. Top/bottom tags are far apart in Z.",
    }),

    make_case("squat_wide", {
        "geometry": {
            "length": 150.0,
            "width":  100.0,
            "height": 10.0,    # very flat
        },
        "_comment": "Low height — nearly 2D. Tests whether gmsh classifies "
                    "top/bottom correctly when they are close in Z.",
    }),

    make_case("thin_wall", {
        "geometry": {
            "length":         100.0,
            "width":           60.0,
            "height":          20.0,
            "wall_thickness":   2.0,   # half the baseline
            "fillet_radius":    1.0,
        },
        "_comment": "Thin walls force small elements in the mesh. "
                    "Exercises aspect ratio checker in Stage 2.",
    }),

    make_case("large_holes", {
        "geometry": {
            "mounting_hole_diameter": 10.0,  # baseline is 6mm
            "mounting_hole_inset":    15.0,
        },
        "_comment": "Larger holes mean more curved surfaces for gmsh to classify. "
                    "Also concentrates stress around hole edges in Stage 3.",
    }),

    # ── Group B: Load variations ──────────────────────────────────────────
    # Tests FEA behavior and safety factor logic across load magnitudes.

    make_case("heavy_load", {
        "load_hints": {
            "load_magnitude_n": 5000.0,   # 5x baseline
        },
        "_comment": "High load — likely pushes safety factor below 1.5 warning "
                    "threshold. Tests Stage 3 warning logic. May fail safety check.",
    }),

    make_case("light_load", {
        "load_hints": {
            "load_magnitude_n": 100.0,    # 10x lower than baseline
        },
        "_comment": "Very low stress throughout. Tests SIMP when the optimizer "
                    "has little signal — density field may not converge to "
                    "a clean binary result.",
    }),

    make_case("load_bottom", {
        "load_hints": {
            "primary_face":     "bottom",
            "load_magnitude_n": 1000.0,
        },
        "_comment": "Load applied to bottom face instead of top. "
                    "Tests boundary_conditions.py face selection logic.",
    }),

    # ── Group C: Mesh resolution variations ───────────────────────────────
    # Tests pipeline performance and quality at different mesh densities.

    make_case("coarse_mesh", {
        "mesh_hints": {
            "target_element_size": 5.0,   # 2.5x baseline — much faster
        },
        "_comment": "Coarse mesh for rapid iteration and smoke testing. "
                    "Lower quality but completes in minutes rather than hours. "
                    "Use this when testing pipeline changes.",
    }),

    make_case("fine_mesh", {
        "mesh_hints": {
            "target_element_size": 1.0,   # 2x finer than baseline
        },
        "_comment": "Fine mesh — tests memory usage and solver scaling. "
                    "Expect 4-8x more elements than baseline. "
                    "Requires at least 16GB WSL2 memory.",
    }),

    # ── Group D: Optimization variations ─────────────────────────────────
    # Tests SIMP convergence behavior and Stage 5 export at extremes.

    make_case("aggressive_optimization", {
        "mesh_hints": {
            "target_element_size": 3.0,   # slightly coarser for speed
        },
        "load_hints": {
            "load_magnitude_n": 1000.0,
        },
        "_comment": "Paired with low VOLUME_FRACTION in pipeline — "
                    "set SIMP_OVERRIDES volume_fraction=0.2 in pipeline_full.ipynb. "
                    "Tests whether SIMP converges cleanly at 20% material retention.",
    }),

    make_case("conservative_optimization", {
        "mesh_hints": {
            "target_element_size": 3.0,
        },
        "_comment": "Paired with high VOLUME_FRACTION in pipeline — "
                    "set SIMP_OVERRIDES volume_fraction=0.7 in pipeline_full.ipynb. "
                    "Tests near-solid result in Stage 5 marching cubes.",
    }),

    # ── Group E: Edge cases ───────────────────────────────────────────────
    # Tests boundary conditions and error handling in the schema validator.

    make_case("minimum_viable", {
        "geometry": {
            "length":                 30.0,
            "width":                  20.0,
            "height":                 10.0,
            "wall_thickness":          2.0,
            "fillet_radius":           1.0,
            "mounting_hole_diameter":  3.0,
            "mounting_hole_inset":     5.0,
        },
        "mesh_hints": {
            "target_element_size": 2.0,
        },
        "load_hints": {
            "load_magnitude_n": 200.0,
        },
        "_comment": "Smallest reasonable part. Tests that the pipeline handles "
                    "small geometry without gmsh producing degenerate elements.",
    }),

    make_case("maximum_size", {
        "geometry": {
            "length": 300.0,
            "width":  200.0,
            "height":  50.0,
            "mounting_hole_inset": 20.0,
        },
        "mesh_hints": {
            "target_element_size": 5.0,   # keep mesh count manageable
        },
        "load_hints": {
            "load_magnitude_n": 10000.0,
        },
        "_comment": "Large part with coarse mesh. Tests that coordinate "
                    "ranges don't cause numerical issues in FEniCSx.",
    }),
]

print(f"Defined {len(TEST_CASES)} test cases")

Defined 14 test cases


Cell 3 — Validate and write all cases

In [4]:
# Cell 3 — Write all cases to disk
print(f"Writing test cases to {output_dir}/\n")

written  = []
skipped  = []
failed   = []

for case in TEST_CASES:
    name = case["part_name"]
    path = write_case(case)
    if path is None:
        failed.append(name)
    elif not path.exists():
        skipped.append(name)
    else:
        written.append(name)

print(f"\n{'─'*40}")
print(f"Written:  {len(written)}")
print(f"Skipped:  {len(skipped)}")
print(f"Failed:   {len(failed)}")
if failed:
    print(f"\nFailed cases (fix before running pipeline):")
    for name in failed:
        print(f"  ✗ {name}")

Writing test cases to test_cases/

  ⏭  baseline — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  tall_narrow — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  squat_wide — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  thin_wall — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  large_holes — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  heavy_load — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  light_load — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  load_bottom — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  coarse_mesh — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  fine_mesh — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  aggressive_optimization — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  conservative_optimization — already exists, skipping (set OVERWRITE=True to replace)
  ⏭  minimum_viable — already exists,

Cell 4 — Preview all cases with OpenSCAD

In [5]:
# Cell 4 — Optionally render each case geometry via OpenSCAD
# Gives a quick visual sanity check that parameters produce valid geometry
# before committing to the full pipeline.
# Set PREVIEW_SCAD = False in Cell 0 to skip.

if not PREVIEW_SCAD:
    print("PREVIEW_SCAD=False — skipping geometry renders")
else:
    import pyvista as pv
    from src.geometry.param_schema import PipelineParams
    from src.geometry.openscad_runner import run_openscad

    pv.OFF_SCREEN = True

    preview_dir = output_dir / "_previews"
    preview_dir.mkdir(exist_ok=True)

    print(f"Rendering {len(written)} geometries via OpenSCAD...\n")

    render_results = {}

    for name in written:
        params_path = output_dir / name / "params.json"
        scad_path   = output_dir / name / "base_part.scad"

        params = PipelineParams.from_json(params_path)

        # Override STL output to go to preview dir
        params.export.stl_output_dir = str(preview_dir)
        params.part_name = name

        result = run_openscad(params, scad_file=scad_path, timeout_s=60)
        render_results[name] = result

        if result.success:
            # Quick headless render
            mesh = pv.read(str(result.stl_path))
            pl = pv.Plotter()
            pl.add_mesh(mesh, color="lightsteelblue", show_edges=False)
            pl.add_axes()
            pl.view_isometric()
            png = preview_dir / f"{name}.png"
            pl.screenshot(str(png))
            pl.close()
            print(f"  ✓ {name:<30} "
                  f"{result.stl_size_kb:>8.1f} KB   "
                  f"{result.duration_s:.1f}s")
        else:
            print(f"  ✗ {name:<30} FAILED: {result.stderr[:80]}")

    n_ok = sum(1 for r in render_results.values() if r.success)
    print(f"\n{n_ok}/{len(written)} geometries rendered successfully")
    print(f"Preview images: {preview_dir}/")

Rendering 14 geometries via OpenSCAD...

  ✓ baseline                          101.8 KB   0.3s
  ✓ tall_narrow                       101.9 KB   0.3s
  ✓ squat_wide                        102.2 KB   0.3s
  ✓ thin_wall                         101.9 KB   0.3s
  ✓ large_holes                       102.2 KB   0.3s
  ✓ heavy_load                        101.8 KB   0.3s
  ✓ light_load                        101.8 KB   0.3s
  ✓ load_bottom                       101.8 KB   0.3s
  ✓ coarse_mesh                       101.8 KB   0.3s
  ✓ fine_mesh                         101.8 KB   0.3s
  ✓ aggressive_optimization           101.8 KB   0.3s
  ✓ conservative_optimization         101.8 KB   0.3s
  ✓ minimum_viable                    102.1 KB   0.3s
  ✓ maximum_size                      102.4 KB   0.3s

14/14 geometries rendered successfully
Preview images: test_cases/_previews/


Cell 5 — Summary table

In [6]:
# Cell 5 — Print a summary table of all cases and their key parameters
# Useful for a quick sanity check and for documentation.

from pathlib import Path
import json

print(f"{'Name':<30} {'L':>6} {'W':>6} {'H':>6} "
      f"{'Wall':>6} {'Mesh':>6} {'Load':>8}  Notes")
print("─" * 100)

for case in TEST_CASES:
    name = case["part_name"]
    g    = case["geometry"]
    m    = case["mesh_hints"]
    l    = case["load_hints"]
    note = case.get("_comment", "")[:40]

    print(
        f"{name:<30} "
        f"{g['length']:>6.0f} "
        f"{g['width']:>6.0f} "
        f"{g['height']:>6.0f} "
        f"{g['wall_thickness']:>6.1f} "
        f"{m['target_element_size']:>6.1f} "
        f"{l['load_magnitude_n']:>8.0f}  "
        f"{note}"
    )

print("─" * 100)
print(f"\nAll params files written to: {output_dir}/")
print("To run a single case through the pipeline:")
print('  papermill notebooks/pipeline_full.ipynb out.ipynb \\')
print(f'    -p PARAMS_JSON "test_cases/baseline/params.json" \\')
print(f'    -p PART_NAME "baseline"')
print("\nTo sweep all cases:")
print("  Edit SWEEP_PARAMS in pipeline_full.ipynb Cell 0 with the list below:\n")

sweep_list = [
    {"PART_NAME": c["part_name"],
     "PARAMS_JSON": f"test_cases/{c['part_name']}/params.json"}
    for c in TEST_CASES
]
print(json.dumps(sweep_list, indent=2))

Name                                L      W      H   Wall   Mesh     Load  Notes
────────────────────────────────────────────────────────────────────────────────────────────────────
baseline                          100     60     20    4.0    8.0    10000  Default parameters for base_part.scad. O
tall_narrow                        60     40     50    4.0    8.0    10000  High aspect ratio geometry — tests surfa
squat_wide                        150    100     10    4.0    8.0    10000  Low height — nearly 2D. Tests whether gm
thin_wall                         100     60     20    2.0    8.0    10000  Thin walls force small elements in the m
large_holes                       100     60     20    4.0    8.0    10000  Larger holes mean more curved surfaces f
heavy_load                        100     60     20    4.0    8.0     5000  High load — likely pushes safety factor 
light_load                        100     60     20    4.0    8.0      100  Very low stress throughout. Tests SIMP 

Cell 6 — Generate the sweep config for pipeline_full.ipynb

In [7]:
# Cell 6 — Write a ready-to-paste sweep config for pipeline_full.ipynb
# Copy the contents of the written file directly into pipeline_full.ipynb
# Cell 0's SWEEP_PARAMS list to run all cases in one make run.

sweep_config_path = output_dir / "sweep_config.json"

sweep_config = {
    "_instructions": (
        "Copy SWEEP_PARAMS into pipeline_full.ipynb Cell 0 "
        "and set SWEEP_MODE = True to run all cases."
    ),
    "SWEEP_MODE": True,
    "SWEEP_PARAMS": [
        {
            "PART_NAME":   case["part_name"],
            "PARAMS_JSON": f"test_cases/{case['part_name']}/params.json",
        }
        for case in TEST_CASES
        if case["part_name"] in written
    ],
    # Sensible per-stage overrides for test runs
    # Override these per-case in SWEEP_PARAMS if needed
    "MESH_OVERRIDES":  {"QUALITY_STRICT": False},  # don't abort sweep on marginal mesh
    "SIMP_OVERRIDES":  {"MAX_ITERATIONS": 50},      # shorter for test runs
    "EXPORT_OVERRIDES":{"VOXEL_RESOLUTION": 80},    # faster export for tests
}

sweep_config_path.write_text(json.dumps(sweep_config, indent=2))
print(f"Sweep config written: {sweep_config_path}")
print("\nContents:")
print(json.dumps(sweep_config, indent=2))

Sweep config written: test_cases/sweep_config.json

Contents:
{
  "_instructions": "Copy SWEEP_PARAMS into pipeline_full.ipynb Cell 0 and set SWEEP_MODE = True to run all cases.",
  "SWEEP_MODE": true,
  "SWEEP_PARAMS": [
    {
      "PART_NAME": "baseline",
      "PARAMS_JSON": "test_cases/baseline/params.json"
    },
    {
      "PART_NAME": "tall_narrow",
      "PARAMS_JSON": "test_cases/tall_narrow/params.json"
    },
    {
      "PART_NAME": "squat_wide",
      "PARAMS_JSON": "test_cases/squat_wide/params.json"
    },
    {
      "PART_NAME": "thin_wall",
      "PARAMS_JSON": "test_cases/thin_wall/params.json"
    },
    {
      "PART_NAME": "large_holes",
      "PARAMS_JSON": "test_cases/large_holes/params.json"
    },
    {
      "PART_NAME": "heavy_load",
      "PARAMS_JSON": "test_cases/heavy_load/params.json"
    },
    {
      "PART_NAME": "light_load",
      "PARAMS_JSON": "test_cases/light_load/params.json"
    },
    {
      "PART_NAME": "load_bottom",
      "PARAMS_JSON"

The output directory structure this produces
After running the notebook you will have:
test_cases/
├── _previews/
│   ├── baseline.png
│   ├── tall_narrow.png
│   ├── squat_wide.png
│   └── ...                          ← one PNG per case
├── baseline/
│   ├── params.json                  ← self-contained params
│   └── base_part.scad               ← copy of scad source
├── tall_narrow/
│   ├── params.json
│   └── base_part.scad
├── thin_wall/
│   ├── params.json
│   └── base_part.scad
├── ...
└── sweep_config.json                ← paste into pipeline_full.ipynb
Each case directory is fully self-contained — you can hand someone just the thin_wall/ folder and they have everything needed to run that case through the pipeline.
